In [2]:
import os
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [ ]:
# Função para carregar os dados rotulados de uma planilha Excel
def load_labeled_data(file_path, sheet_name):
    labeled_data = pd.read_excel(file_path, sheet_name=sheet_name)
    return labeled_data

# Função para carregar os dados não rotulados de um arquivo CSV
def load_unlabeled_data(file_path):
    unlabeled_data = pd.read_csv(file_path)
    return unlabeled_data

# Função para rotular os textos não rotulados usando similaridade de cosseno
def label_unlabeled_data(unlabeled_data, labeled_data, vectorizer, threshold=0.9):
    X_unlabeled_tfidf = vectorizer.transform(unlabeled_data['text'])
    X_labeled_tfidf = vectorizer.transform(labeled_data['text'])
    labels = []
    similarities = []
    similar_terms = []

    for i in range(X_unlabeled_tfidf.shape[0]):
        similarity_scores = cosine_similarity(X_unlabeled_tfidf[i], X_labeled_tfidf)
        max_similarity = similarity_scores.max()
        similarities.append(max_similarity)

        if max_similarity > threshold:
            similar_doc_index = similarity_scores.argmax()
            labels.append(labeled_data.iloc[similar_doc_index]['desinformacao'])
            similar_terms.append(labeled_data.iloc[similar_doc_index]['termo'])  # Pega o termo associado
        else:
            labels.append(-1)
            similar_terms.append(None)

    return labels, similarities, similar_terms

# Definindo os caminhos dos arquivos para cada mês
file_paths = {
    'Nov_2022': {
        'labeled': 'C://Users//pc//Desktop//Pdpd_2023//Nov_2022//texto_encontrados_nov_2022.xlsx',
        'unlabeled': 'C://Users//pc//Desktop//Pdpd_2023//Nov_2022//clusters_RTs//Nov_2022_cluster_0.csv'
    }
     'Nov_2022': {
        'labeled': 'C://Users//pc//Desktop//Pdpd_2023//Oct_2022//texto_encontrados_oct_2022.xlsx',
        'unlabeled': 'C://Users//pc//Desktop//Pdpd_2023//Oct_2022//clusters_RTs//Oct_2022_cluster_0.csv'
    },
    'Sep_2022': {
        'labeled': 'C://Users//pc//Desktop//Pdpd_2023//Sep_2022//texto_encontrados_sep_2022.xlsx',
        'unlabeled': 'C://Users//pc//Desktop//Pdpd_2023//Sep_2022//clusters_RTs//Set_2022_cluster_0.csv'
    },
    'Aug_2022': {
        'labeled': 'C://Users//pc//Desktop//Pdpd_2023//Aug_2022//texto_encontrados_aug_2022.xlsx',
        'unlabeled': 'C://Users//pc//Desktop//Pdpd_2023//Aug_2022//clusters_RTs//Aug_2022_cluster_0.csv'
    },
    'Jul_2022': {
        'labeled': 'C://Users//pc//Desktop//Pdpd_2023//Jul_2022//texto_encontrados_jul_2022.xlsx',
        'unlabeled': 'C://Users//pc//Desktop//Pdpd_2023//Jul_2022//clusters_RTs//Jul_2022_cluster_0.csv'
    },
    'Jun_2022': {
        'labeled': 'C://Users//pc//Desktop//Pdpd_2023//Jun_2022//texto_encontrados_jun_2022.xlsx',
        'unlabeled': 'C://Users//pc//Desktop//Pdpd_2023//Jun_2022//clusters_RTs//Jun_2022_cluster_0.csv'
    },
    'May_2022': {
        'labeled': 'C://Users//pc//Desktop//Pdpd_2023//May_2022//texto_encontrados_may_2022.xlsx',
        'unlabeled': 'C://Users//pc//Desktop//Pdpd_2023//May_2022//clusters_RTs//May_2022_cluster_0.csv'
    },
    'Apr_2022': {
        'labeled': 'C://Users//pc//Desktop//Pdpd_2023//Apr_2022//texto_encontrados_apr_2022.xlsx',
        'unlabeled': 'C://Users//pc//Desktop//Pdpd_2023//Apr_2022//clusters_RTs//Apr_2022_cluster_0.csv'
    },
    'Mar_2022': {
        'labeled': 'C://Users//pc//Desktop//Pdpd_2023//Mar_2022//texto_encontrados_mar_2022.xlsx',
        'unlabeled': 'C://Users//pc//Desktop//Pdpd_2023//Mar_2022//clusters_RTs//Mar_2022_cluster_0.csv'
    },
    'Feb_2022': {
        'labeled': 'C://Users//pc//Desktop//Pdpd_2023//Fev_2022//texto_encontrados_fev_2022.xlsx',
        'unlabeled': 'C://Users//pc//Desktop//Pdpd_2023//Fev_2022//clusters_RTs//Fev_2022_cluster_0.csv'
    }
}

# Iterar sobre os meses e processar os dados
for month, paths in file_paths.items():
    # Carregar os dados rotulados e não rotulados
    labeled_data = load_labeled_data(paths['labeled'], 'Cluster_0')
    unlabeled_data = load_unlabeled_data(paths['unlabeled'])

    # Vetorizar os textos rotulados com TF-IDF
    vectorizer = TfidfVectorizer().fit(labeled_data['text'])

    # Rotular os textos não rotulados e obter as similaridades e termos
    labels, similarities, similar_terms = label_unlabeled_data(unlabeled_data, labeled_data, vectorizer)

    # Adicionar os rótulos, similaridades e termos ao DataFrame não rotulado
    unlabeled_data['desinformacao'] = labels
    unlabeled_data['similaridade'] = similarities
    unlabeled_data['termo_similar'] = similar_terms

    # Definir o caminho de saída para o mês atual
    output_path = f'C://Users//pc//Desktop//Pdpd_2023//{month}//rotulado//{month}_cluster_0_labeled.xlsx'

    # Salvar todos os dados rotulados e não rotulados em um novo Excel
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        unlabeled_data.to_excel(writer, sheet_name='Rotulados', index=False)

        # Filtrar os dados que não foram rotulados (-1)
        unlabeled_unlabeled = unlabeled_data[unlabeled_data['desinformacao'] == -1]
        unlabeled_unlabeled.to_excel(writer, sheet_name='Não Rotulados', index=False)

    print(f"Dados salvos em: {output_path}")


In [ ]:
# Função para carregar os dados rotulados de uma planilha Excel
def load_labeled_data(file_path, sheet_name):
    labeled_data = pd.read_excel(file_path, sheet_name=sheet_name)
    return labeled_data

# Função para carregar os dados não rotulados de um arquivo CSV
def load_unlabeled_data(file_path):
    unlabeled_data = pd.read_csv(file_path)
    return unlabeled_data

# Função para rotular os textos não rotulados usando similaridade de cosseno
def label_unlabeled_data(unlabeled_data, labeled_data, vectorizer, threshold=0.9):
    X_unlabeled_tfidf = vectorizer.transform(unlabeled_data['text'])
    X_labeled_tfidf = vectorizer.transform(labeled_data['text'])
    labels = []
    similarities = []
    similar_terms = []

    for i in range(X_unlabeled_tfidf.shape[0]):
        similarity_scores = cosine_similarity(X_unlabeled_tfidf[i], X_labeled_tfidf)
        max_similarity = similarity_scores.max()
        similarities.append(max_similarity)

        if max_similarity > threshold:
            similar_doc_index = similarity_scores.argmax()
            labels.append(labeled_data.iloc[similar_doc_index]['desinformacao'])
            similar_terms.append(labeled_data.iloc[similar_doc_index]['termo'])  # Pega o termo associado
        else:
            labels.append(-1)
            similar_terms.append(None)

    return labels, similarities, similar_terms

# Definindo os caminhos dos arquivos para cada mês
file_paths = {
    'Nov_2022': {
        'labeled': 'C://Users//pc//Desktop//Pdpd_2023//Nov_2022//texto_encontrados_nov_2022.xlsx',
        'unlabeled': 'C://Users//pc//Desktop//Pdpd_2023//Nov_2022//clusters_RTs//Nov_2022_cluster_1.csv'
    }
     'Nov_2022': {
        'labeled': 'C://Users//pc//Desktop//Pdpd_2023//Oct_2022//texto_encontrados_oct_2022.xlsx',
        'unlabeled': 'C://Users//pc//Desktop//Pdpd_2023//Oct_2022//clusters_RTs//Oct_2022_cluster_1.csv'
    },
    'Sep_2022': {
        'labeled': 'C://Users//pc//Desktop//Pdpd_2023//Sep_2022//texto_encontrados_sep_2022.xlsx',
        'unlabeled': 'C://Users//pc//Desktop//Pdpd_2023//Sep_2022//clusters_RTs//Set_2022_cluster_1.csv'
    },
    'Aug_2022': {
        'labeled': 'C://Users//pc//Desktop//Pdpd_2023//Aug_2022//texto_encontrados_aug_2022.xlsx',
        'unlabeled': 'C://Users//pc//Desktop//Pdpd_2023//Aug_2022//clusters_RTs//Aug_2022_cluster_1.csv'
    },
    'Jul_2022': {
        'labeled': 'C://Users//pc//Desktop//Pdpd_2023//Jul_2022//texto_encontrados_jul_2022.xlsx',
        'unlabeled': 'C://Users//pc//Desktop//Pdpd_2023//Jul_2022//clusters_RTs//Jul_2022_cluster_1.csv'
    },
    'Jun_2022': {
        'labeled': 'C://Users//pc//Desktop//Pdpd_2023//Jun_2022//texto_encontrados_jun_2022.xlsx',
        'unlabeled': 'C://Users//pc//Desktop//Pdpd_2023//Jun_2022//clusters_RTs//Jun_2022_cluster_1.csv'
    },
    'May_2022': {
        'labeled': 'C://Users//pc//Desktop//Pdpd_2023//May_2022//texto_encontrados_may_2022.xlsx',
        'unlabeled': 'C://Users//pc//Desktop//Pdpd_2023//May_2022//clusters_RTs//May_2022_cluster_1.csv'
    },
    'Apr_2022': {
        'labeled': 'C://Users//pc//Desktop//Pdpd_2023//Apr_2022//texto_encontrados_apr_2022.xlsx',
        'unlabeled': 'C://Users//pc//Desktop//Pdpd_2023//Apr_2022//clusters_RTs//Apr_2022_cluster_1.csv'
    },
    'Mar_2022': {
        'labeled': 'C://Users//pc//Desktop//Pdpd_2023//Mar_2022//texto_encontrados_mar_2022.xlsx',
        'unlabeled': 'C://Users//pc//Desktop//Pdpd_2023//Mar_2022//clusters_RTs//Mar_2022_cluster_1.csv'
    },
    'Feb_2022': {
        'labeled': 'C://Users//pc//Desktop//Pdpd_2023//Fev_2022//texto_encontrados_fev_2022.xlsx',
        'unlabeled': 'C://Users//pc//Desktop//Pdpd_2023//Fev_2022//clusters_RTs//Fev_2022_cluster_1.csv'
    }
}

# Iterar sobre os meses e processar os dados
for month, paths in file_paths.items():
    # Carregar os dados rotulados e não rotulados
    labeled_data = load_labeled_data(paths['labeled'], 'Cluster_1')
    unlabeled_data = load_unlabeled_data(paths['unlabeled'])

    # Vetorizar os textos rotulados com TF-IDF
    vectorizer = TfidfVectorizer().fit(labeled_data['text'])

    # Rotular os textos não rotulados e obter as similaridades e termos
    labels, similarities, similar_terms = label_unlabeled_data(unlabeled_data, labeled_data, vectorizer)

    # Adicionar os rótulos, similaridades e termos ao DataFrame não rotulado
    unlabeled_data['desinformacao'] = labels
    unlabeled_data['similaridade'] = similarities
    unlabeled_data['termo_similar'] = similar_terms

    # Definir o caminho de saída para o mês atual
    output_path = f'C://Users//pc//Desktop//Pdpd_2023//{month}//rotulado//{month}_cluster_1_labeled.xlsx'

    # Salvar todos os dados rotulados e não rotulados em um novo Excel
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        unlabeled_data.to_excel(writer, sheet_name='Rotulados', index=False)

        # Filtrar os dados que não foram rotulados (-1)
        unlabeled_unlabeled = unlabeled_data[unlabeled_data['desinformacao'] == -1]
        unlabeled_unlabeled.to_excel(writer, sheet_name='Não Rotulados', index=False)

    print(f"Dados salvos em: {output_path}")
